In [28]:
from pyaraucaria.obs_plan.obs_plan_parser import ObsPlanParser
from jsonschema import Draft7Validator

In [279]:
def convert_to_obdict(ob):
    result = {"name": None, "ra": None, "dec": None, "type": None}

    subcommands = ob.get("subcommands", [])
    if isinstance(subcommands, dict):
        subcommands = [subcommands]

    for sub in subcommands:
        if "command_name" in sub:
            result["type"] = sub["command_name"]
        if "kwargs" in sub and isinstance(sub["kwargs"], dict):
            result.update(sub["kwargs"])
        if "args" in sub and isinstance(sub["args"], list):
            if len(sub["args"]) == 1:
                result["name"] = sub["args"][0]
            elif len(sub["args"]) == 3:
                result["name"] = sub["args"][0]
                result["ra"] = sub["args"][1]
                result["dec"] = sub["args"][2]

    return result



In [280]:
WAIT_SCHEMA = {
    "type": "object",
    "required": ["type"],  # w Twoim płaskim słowniku 'type' = 'WAIT'
    "properties": {
        "type": {"const": "WAIT"},
        "sec": {"type": "number", "minimum": 0},
        "ut": {"type": "string", "pattern": r"^\d{1,2}:\d{2}:\d{2}$"},
        "sunrise": {"type": "number"},
        "sunset": {"type": "number"},
        "observer": {"type": "string"},
        "uobi": {"type": "string"},
    },
    "additionalProperties": True,
}

In [281]:
ZERO_SCHEMA = {
    "type": "object",
    "required": ["type", "seq"],
    "properties": {

        "type": {"const": "ZERO"},

        # args
        "name": {"type": ["string", "null"]},

        # kwargs
        "seq": {"type": "string"},
        # UWAGA: w OCM seq może mieć postać:
        # 5
        # 5/Ic/0,5/V/0
        # więc traktujemy jako string (parsowanie robisz później)

        "mirror_cover": {
            "type": "string",
            "enum": ["open", "close"]
        },

        "read_mod": {"type": "string"},

        "test": {"type": "boolean"},

        "focus_offset": {"type": "number"},

        "observer": {"type": "string"},
        "uobi": {"type": "integer"},
        "sciprog": {"type": "string"},
        "pi": {"type": "string"}
    },

    "additionalProperties": True
}

In [282]:
DARK_SCHEMA = {
    "type": "object",
    "required": ["type", "seq"],
    "properties": {

        "type": {"const": "DARK"},

        # args
        "name": {"type": ["string", "null"]},

        # w OCM seq może być złożone (jak w ZERO)
        "seq": {
            "type": "string"
        },

        "read_mod": {"type": "string"},

        "test": {"type": "boolean"},

        "focus_offset": {"type": "number"},

        "observer": {"type": "string"},
        "uobi": {"type": "integer"},
        "sciprog": {"type": "string"},
        "pi": {"type": "string"}
    },

    "additionalProperties": True
}

In [283]:
DOMEFLAT_SCHEMA = {
    "type": "object",
    "required": ["seq"],
    "properties": {

        "type": {"const": "DOMEFLAT"},

        # args
        "name": {"type": ["string", "null"]},

        # kwargs
        "seq": {
            "type": "string"
        },

        "lamp": {
            "type": "string"
        },

        "read_mod": {"type": "string"},

        "test": {"type": "boolean"},

        "focus_offset": {"type": "number"},

        "observer": {"type": "string"},
        "uobi": {"type": "integer"},
        "sciprog": {"type": "string"},
        "pi": {"type": "string"}
    },

    "additionalProperties": True
}

In [284]:
SKYFLAT_SCHEMA = {
    "type": "object",
    "required": ["type", "seq"],
    "properties": {

        "type": {"const": "SKYFLAT"},

        # args
        "name": {"type": ["string", "null"]},

        # kwargs
        "seq": {
            "type": "string"
        },

        "read_mod": {"type": "string"},

        "test": {"type": "boolean"},

        "focus_offset": {"type": "number"},

        "observer": {"type": "string"},
        "uobi": {"type": "integer"},
        "sciprog": {"type": "string"},
        "pi": {"type": "string"}
    },

    "additionalProperties": True
}

In [285]:
FOCUS_SCHEMA = {
    "type": "object",
    "required": ["type","seq"],  # object_name i kwargs są opcjonalne
    "properties": {
        "type": {"const": "FOCUS"},

        # args
        "name": {"type": ["string", "null"]},
        "ra": {"type": ["string", "null"]},
        "dec": {"type": ["string", "null"]},

        # kwargs
        "seq": {"type": "string"},
        "az": {"type": "number"},
        "alt": {"type": "number"},
        "pos": {"type": "string"},
        "auto_focus": {"type": "string", "enum": ["on", "off"]},
        "tracking": {"type": "string", "enum": ["on", "off"]},
        "dome_follow": {"type": "string", "enum": ["off"]},
        "mirror_cover": {"type": "string", "enum": ["auto", "open", "close"]},
        "read_mod": {"type": ["integer", "string"]},
        "test": {"type": "boolean"},
        "epoch": {"type": "integer"},
        "observer": {"type": "string"},
        "uobi": {"type": "integer"}
    },
    "additionalProperties": True
}

In [331]:
OBJECT_SCHEMA = {
    "type": "object",
    "required": ["type"],  # object_name i kwargs są opcjonalne
    "oneOf": [{"required": ["ra", "dec"]},{"required": ["alt", "az"]}],
    "properties": {
        "type": {"const": "OBJECT"},

        # args
        "name": {"type": ["string", "null"]},
        "ra": {"type": ["string", "null"]},
        "dec": {"type": ["string", "null"]},

        # kwargs
        "seq": {"type": "string"},
        "az": {"type": "number"},
        "alt": {"type": "number"},
        "tracking": {"type": "string", "enum": ["on", "off"]},
        "dither": {"type": "string"},
        "dome_follow": {"type": "string", "enum": ["off"]},
        "focus_offset": {"type": ["string", "boolean"]},
        "mirror_cover": {"type": "string", "enum": ["auto", "open", "close"]},
        "epoch": {"type": "integer"},
        "read_mod": {"type": ["integer", "string"]},
        "test": {"type": "boolean"},
        "observer": {"type": "string"},
        "uobi": {"type": "integer"},
        "sciprog": {"type": "string"},
        "pi": {"type": "string"}
    },
    "additionalProperties": False
}

In [332]:


def convert_and_validate(obs):

    ob = {k: v for k, v in obs.items() if v is not None}

    if ob.get("type") == "WAIT":
        schema = WAIT_SCHEMA
    elif ob.get("type") == "ZERO":
        schema = ZERO_SCHEMA
    elif ob.get("type") == "DARK":
        schema = DARK_SCHEMA
    elif ob.get("type") == "DOMEFLAT":
        schema = DOMEFLAT_SCHEMA
    elif ob.get("type") == "SKYFLAT":
        schema = SKYFLAT_SCHEMA
    elif ob.get("type") == "FOCUS":
        schema = FOCUS_SCHEMA
    elif ob.get("type") == "OBJECT":
        schema = OBJECT_SCHEMA

    validator = Draft7Validator(schema)

    # słownik wynikowy: wszystkie możliwe properties ze schema
    result = {key: True for key in schema.get("properties", {})}

    # --- 1. Konwersja typów ---
    for key, prop in schema.get("properties", {}).items():
        if key not in ob:
            continue
        if "type" not in prop:
            continue

        val = ob[key]
        typ = prop["type"]

        try:
            if typ == "number":
                ob[key] = float(val)
            elif typ == "integer":
                ob[key] = int(val)
        except (ValueError, TypeError):
            result[key] = False

    # --- 2. Walidacja JSON Schema ---
    for error in validator.iter_errors(ob):
        print(error.message)
        if error.validator == "required":
            # błąd wymaganych pól
            for req in error.validator_value:  # lista required z schematu
                result[req] = None
        elif error.validator == "oneOf":
            # błąd oneOf → wszystkie pola wymagane w każdym podschemacie oznaczamy False
            for subschema in error.validator_value:
                for key in subschema.get("required", []):
                    result[key] = False
        else:
            # inne błędy przypisane do konkretnego pola
            if error.path:
                key = error.path[0]
                result[key] = False

    # --- 3. Logical validation ---
    if ob.get("type") == "WAIT":
        if not any(ob.get(k) is not None for k in ["sec", "ut", "sunrise", "sunset"]):
            for k in ["sec", "ut", "sunrise", "sunset"]:
                result[k] = False

    return result


In [334]:
# txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 seq=1/V/20 \n WAIT ut=16:00:00"
# txt = "WAIT ut=16:00:00"
# txt = "WAIT sunrise=-10"
# txt = "WAIT sec=600"
# txt = "ZERO seq=15/Ic/0"
# txt = "DARK ZZ01 seq=10/V/300,10/Ic/200"
# txt = "DOMEFLAT AL3 seq=10/i/100"
# txt = "SKYFLAT HD23 alt=60.0 az=270.0 seq=10/r/20,10/V/30"
# txt = "SKYFLAT HD24 seq=10/g/a,10/V/a"
# txt = "FOCUS NG31 12:12:12 20:20:20"
# txt = "OBJECT HD193901 20:23:35.8 -21:22:14.0 seq=1/V/300"
# txt = "OBJECT FF_Aql 18:58:14.75 17:21:39.29 seq=5/Ic/60,5/V/70"
txt = "OBJECT V496_Aql 19:08:20.77 -07:26:15.89 seq=1/V/20"
# txt = "OBJECT HD23 alt=dupa az=270.0"
# txt = "OBJECT V496_Aql 19:08:20.77 "

ob_tmp = ObsPlanParser.convert_from_string(txt)
#print(ob_tmp)
ob = convert_to_obdict(ob_tmp)
#print(ob)
convert_and_validate(ob)

{'type': True,
 'name': True,
 'ra': True,
 'dec': True,
 'seq': True,
 'az': True,
 'alt': True,
 'tracking': True,
 'dither': True,
 'dome_follow': True,
 'focus_offset': True,
 'mirror_cover': True,
 'epoch': True,
 'read_mod': True,
 'test': True,
 'observer': True,
 'uobi': True,
 'sciprog': True,
 'pi': True}